In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# yollar

DATA_DIR = Path("../data")
FIG_DIR = Path("../src")

pd.set_option("display.max_columns", 50)

df = pd.read_parquet(DATA_DIR / "train_merged.parquet")

# zamana göre train ve test setleri ayıracağımız için kronolojik olarak sıralıyoruz

df.sort_values("TransactionDT", kind="mergesort").reset_index(drop=True)

df["tx_day"] = (df["TransactionDT"] // 86400).astype("int16")  # 1 gün = 86400 saniye
print("Gün aralığı:", df["tx_day"].min(), "-", df["tx_day"].max())


Gün aralığı: 1 - 182


In [5]:
# --- Ayrım noktasını belirle ---
# Hedef: işlemlerin yaklaşık son %20'si validation olsun.
# Satır sayısına göre quantile alıyoruz, sonra GÜN sınırına yuvarlıyoruz.
# Böylece aynı gün hem train'e hem validation'a bölünmüyor.

VALIDATION_FRAC = 0.2
cutoff_day = int(np.quantile(df["tx_day"], 1 - VALIDATION_FRAC))

# True = validation, False = train
df["is_valid"] = df["tx_day"] >= cutoff_day

# --- Özet tablo: iki tarafın boyutu, gün aralığı ve fraud oranı ---

split_summary = (
    df.groupby("is_valid")
        .agg(n_rows=("isFraud", "size"), 
                day_min=("tx_day", "min"), 
                day_max=("tx_day", "max"), 
                fraud_rate=("isFraud", "mean"))
            .rename(index={False: "train", True: "validation"})
)

split_summary["row_share"] = split_summary["n_rows"] / len(df)
print("Ayrım günü:", cutoff_day)
split_summary.style.format({
    "fraud_rate": "{:.2%}",
    "row_share": "{:.1%}"
})

Ayrım günü: 141


,n_rows,day_min,day_max,fraud_rate,row_share
is_valid,,,,,
train,472004,1,140,3.51%,79.9%
validation,118533,141,182,3.44%,20.1%


In [6]:
# Satır bazlı özellikler — bayraklar, zaman, tutar

# --- 1) Eksiklik bayrakları ---

# `id_01`, identity tablosunda **hiç boş olmayan** bir sütun. 
# Join sonrası NaN ise bunun tek açıklaması identity kaydının olmaması → güvenilir bir eşleşme işareti.

df["has_identity"] = df["id_01"].notna().astype("int8")

# 02'de `addr1` ve `addr2`'nin **her zaman birlikte boş** olduğu görüldü, dolayısıyla tek sütun yeterli.
# - `addr1` seçildi çünkü daha ayrıntılı (bölge düzeyindeki) sütun.

df["has_addr"] = df["addr1"].notna().astype("int8")

In [7]:
# --- 2) Zaman özellikleri ---

df["tx_hour"] = ((df["TransactionDT"] // 3600) % 24).astype("int8") # 1 saat = 3600 saniye

# veri başlangıcından itibaren 7'lik döngü

df["tx_weekday"] = (df["tx_day"] % 7).astype("int8")

In [8]:
# --- 3) Tutar özellikleri ---

amt = df["TransactionAmt"].astype("float64")

df["amt_log"] = np.log1p(amt).astype("float32")

amt_milli = np.round(amt * 1000).astype("int64") # Kuruş kısmı: tutarı 1000 ile çarpıp tamsayıya yuvarlıyoruz

# amt_cents: ondalık kısım, kuruş cinsinden (0-99)
df["amt_cents"] = ((amt_milli % 1000) // 10).astype("int16")

df["amt_n_decimals"] = np.select(
    [amt_milli % 1000 == 0,     # tam sayı: 49.000
     amt_milli % 100 == 0,      # 1 basamak: 49.500
     amt_milli % 10 == 0],      # 2 basamak: 49.950
    [0, 1, 2],
    default=3                   # 3 basamak: 49.957
).astype("int8")


In [9]:
# Feature olmayan sütunlar
NON_FEATURES = ["TransactionID", "isFraud", "TransactionDT", "tx_day", "is_valid"]

# Kısayol maskeleri: istatistikleri SADECE train'den hesaplamak için
train_mask = ~df["is_valid"]
valid_mask = df["is_valid"]

print("train:", train_mask.sum(), "| validation:", valid_mask.sum())
# beklenen: train: 472004 | validation: 118533

train: 472004 | validation: 118533


In [10]:
# --- 4) Kontrol: yeni özelliklerin fraud oranına etkisi (SADECE train) ---
# Amaç modellemek değil, özelliklerin mantıklı davrandığını görmek.
# Validation'a bakmıyoruz; karar verirken onu görmemiş olmamız lazım.
new_cols = ["has_identity", "has_addr", "amt_n_decimals"]

for col in new_cols:
    summary = (
        df.loc[train_mask]
          .groupby(col)["isFraud"]
          .agg(n="size", fraud_rate="mean")
    )
    summary["share"] = summary["n"] / summary["n"].sum()
    print(f"\n--- {col} ---")
    print(summary.to_string(formatters={"fraud_rate": "{:.2%}".format,
                                        "share": "{:.1%}".format}))


--- has_identity ---
                   n fraud_rate share
has_identity                         
0             351634      2.13% 74.5%
1             120370      7.55% 25.5%

--- has_addr ---
               n fraud_rate share
has_addr                         
0          53703     11.40% 11.4%
1         418301      2.50% 88.6%

--- amt_n_decimals ---
                     n fraud_rate share
amt_n_decimals                         
0               244481      3.62% 51.8%
1                18670      1.92%  4.0%
2               157927      1.03% 33.5%
3                50926     11.33% 10.8%


In [11]:
# amt_n_decimals'ın etkisi ürünle mi karışıyor? (SADECE train)
tr = df.loc[train_mask]

# 1) Her ürünün içinde ondalık basamak dağılımı (satır yüzdesi)
#    3 basamaklılar neredeyse tamamen C'de toplanıyorsa → ürün etkisi
print("Ürün içi dağılım:")
print(pd.crosstab(tr["ProductCD"], tr["amt_n_decimals"], normalize="index")
        .map("{:.1%}".format))

# 2) Ürün içinde fraud oranı
#    Aynı ürün içinde basamak sayısına göre oran değişiyorsa → bağımsız sinyal
print("\nÜrün içi fraud oranı:")
print(tr.pivot_table(index="ProductCD", columns="amt_n_decimals",
                     values="isFraud", aggfunc="mean", observed=True)
        .map(lambda x: f"{x:.2%}" if pd.notna(x) else "-"))

Ürün içi dağılım:
amt_n_decimals       0     1      2      3
ProductCD                                 
C                 0.1%  0.6%   8.9%  90.4%
H               100.0%  0.0%   0.0%   0.0%
R               100.0%  0.0%   0.0%   0.0%
S                99.5%  0.0%   0.5%   0.0%
W                50.5%  5.3%  44.2%   0.0%

Ürün içi fraud oranı:
amt_n_decimals       0       1       2       3
ProductCD                                     
C               41.38%  11.51%  11.10%  11.33%
H                4.64%       -       -       -
R                3.63%       -       -       -
S                6.23%       -   0.00%       -
W                3.31%   1.73%   0.70%   0.00%


In [12]:

# E-posta özellikleri

# Sağlayıcı tipleri. Listeler veride sık görülen domain'lerden derlendi;
# kapsamlı değil, listede olmayanlar "other" olacak.
WEBMAIL = {"gmail", "yahoo", "hotmail", "outlook", "live", "msn", "aol",
           "icloud", "me", "mac", "mail", "ymail", "protonmail", "gmx", "yandex"}
ISP     = {"att", "verizon", "sbcglobal", "comcast", "charter", "cox",
           "bellsouth", "earthlink", "centurylink", "frontier", "windstream",
           "optonline", "roadrunner", "twc", "embarqmail", "juno", "netzero",
           "cfl", "suddenlink", "frontiernet", "q", "prodigy"}
ANON    = {"anonymous"}


def add_email_features(df, col, prefix):
    """col sütunundan (örn. 'P_emaildomain') sağlayıcı, tip ve uzantı üretir.
    NaN'lar korunur: domain yoksa yeni sütunlar da NaN olur."""

    # category → object: .str metotları NaN'ı NaN olarak bırakır
    # (category'de .astype(str) ile "nan" metnine dönüşme sorununa girmiyoruz)
    s = df[col].astype("object").str.lower().str.strip()
    parts = s.str.split(".")

    # 1) Sağlayıcı: ilk parça. "gmail.com" ve "gmail" → "gmail",
    #    "hotmail.co.uk" → "hotmail", "yahoo.com.mx" → "yahoo"
    provider = parts.str[0]
    df[f"{prefix}_email_provider"] = provider.astype("category")

    # 2) Sağlayıcı tipi: webmail / isp / anonymous / other
    ptype = pd.Series(
        np.select([provider.isin(WEBMAIL), provider.isin(ISP), provider.isin(ANON)],
                  ["webmail", "isp", "anonymous"],
                  default="other"),
        index=df.index,
    ).where(provider.notna())          # domain yoksa NaN kalsın, "other" olmasın
    df[f"{prefix}_email_type"] = ptype.astype("category")

    # 3) Uzantı: son parça ("com", "net", "mx", "uk"...).
    #    Noktasız domain'lerde ("gmail") uzantı bilinmiyor → NaN
    suffix = parts.str[-1].where(parts.str.len() > 1)
    df[f"{prefix}_email_suffix"] = suffix.astype("category")

    return df


df = add_email_features(df, "P_emaildomain", "P")
df = add_email_features(df, "R_emaildomain", "R")

# 4) P == R bayrağı: sadece İKİSİ DE doluysa anlamlı.
#    Biri boşsa "eşit değil" demek yanlış olur → NaN bırakıyoruz.
#    İki farklı category sütunu doğrudan karşılaştırılamaz (kategori listeleri
#    farklı olduğu için hata verir), bu yüzden object'e çeviriyoruz.
p = df["P_emaildomain"].astype("object")
r = df["R_emaildomain"].astype("object")
both = p.notna() & r.notna()
df["email_match"] = np.where(both, (p == r).astype("float32"), np.nan).astype("float32")

In [13]:
# --- Kontrol (SADECE train) ---
tr = df.loc[train_mask]

def rate_table(col, min_n=1000):
    """Sütunun değerlerine göre işlem sayısı ve fraud oranı.
    Küçük gruplar (min_n altı) gürültülü olduğu için gösterilmez."""
    t = (tr.groupby(col, observed=True, dropna=False)["isFraud"]
           .agg(n="size", fraud_rate="mean")
           .query("n >= @min_n")
           .sort_values("n", ascending=False))
    print(f"\n--- {col} ---")
    print(t.to_string(formatters={"fraud_rate": "{:.2%}".format}))

rate_table("P_email_type")
rate_table("email_match")
rate_table("P_email_suffix")

# Sağlayıcı tipinin etkisi W içinde de sürüyor mu? (ürün karışıklığı kontrolü)
print("\nÜrün içi fraud oranı (P_email_type):")
print(tr.pivot_table(index="ProductCD", columns="P_email_type", values="isFraud",
                     aggfunc="mean", observed=True)
        .map(lambda x: f"{x:.2%}" if pd.notna(x) else "-"))


--- P_email_type ---
                   n fraud_rate
P_email_type                   
webmail       345739      3.84%
NaN            73524      2.99%
anonymous      29943      2.33%
isp            21481      1.71%
other           1317      3.49%

--- email_match ---
                  n fraud_rate
email_match                   
NaN          366112      2.20%
1.0           85781      9.29%
0.0           20111      2.82%

--- P_email_suffix ---
                     n fraud_rate
P_email_suffix                   
com             374486      3.72%
NaN              73934      2.98%
net              19869      1.78%
mx                2087      1.63%

Ürün içi fraud oranı (P_email_type):
P_email_type anonymous    isp  other webmail
ProductCD                                   
C                6.51%  0.49%  0.00%  11.78%
H                3.98%  1.90%  0.00%   5.05%
R                1.07%  0.73%  1.56%   5.03%
W                1.30%  1.89%  4.77%   2.08%


In [14]:
# Ürün içinde hem işlem sayısı hem fraud oranı (SADECE train)
# Oranlar ancak grup yeterince büyükse anlamlı: n'ye bakmadan yorum yapmıyoruz.
def within_product(col, min_n=300):
    t = (tr.groupby(["ProductCD", col], observed=True, dropna=False)["isFraud"]
           .agg(n="size", fraud_rate="mean")
           .query("n >= @min_n"))          # küçük grupları gizle
    print(f"\n--- {col} (ürün içinde, n >= {min_n}) ---")
    print(t.to_string(formatters={"fraud_rate": "{:.2%}".format}))

within_product("email_match")
within_product("P_email_type")


--- email_match (ürün içinde, n >= 300) ---
                            n fraud_rate
ProductCD email_match                   
C         1.0           54240     11.23%
          NaN            2112     13.87%
H         0.0            7855      3.18%
          1.0           11631      8.80%
          NaN           10187      1.01%
R         0.0           12256      2.59%
          1.0           19910      4.27%
S         NaN            8118      6.20%
W         NaN          345667      2.07%

--- P_email_type (ürün içinde, n >= 300) ---
                             n fraud_rate
ProductCD P_email_type                   
C         anonymous       4762      6.51%
          webmail        49051     11.78%
          NaN             2112     13.87%
H         anonymous       2913      3.98%
          isp             2686      1.90%
          webmail        23978      5.05%
R         anonymous       8009      1.07%
          isp             3029      0.73%
          webmail        21064      5.